In [48]:
from __future__ import annotations

import os
import re
import json
from datetime import date, datetime
from decimal import Decimal
from pathlib import Path
from typing import Any, Dict, List, Literal, Optional, Tuple, TypedDict
from langchain.chat_models import init_chat_model

from dotenv import load_dotenv

import pandas as pd
import pdfplumber
from pydantic import BaseModel, Field, ValidationError

from langgraph.graph import StateGraph, START, END
from langchain_core.messages import SystemMessage, HumanMessage
from IPython.display import Markdown, display


load_dotenv(override=True)


True

In [49]:
# ============================================================
# 1) LLM 설정
# ============================================================
LLM_MODEL = os.getenv("LLM_MODEL")
LLM_BASE_URL=os.getenv("LLM_BASE_URL")
LLM_API_KEY=os.getenv("LLM_API_KEY")

_SYSTEM_PROMPT = (
    "너는 자산운용사 변액일임펀드 설정/해지 지시서에서 "
    "DB 적재용 구조화 데이터를 추출하는 전문가다. "
    "추측하지 말고, 문서 근거가 없는 값은 만들지 마라."
)

In [50]:
def create_llm_model_(max_tokens: int = 2048):
    """
    vLLM(OpenAI 호환) + Qwen 모델을 LangChain init_chat_model로 래핑
    - max_tokens를 걸어 출력 폭증 방지
    """
    llm = init_chat_model(
        f"openai:{LLM_MODEL}",
        temperature=0.0,
        top_p=0.1,
        base_url=LLM_BASE_URL,
        api_key=LLM_API_KEY,
        model_kwargs={"max_tokens": max_tokens},
    )
    return llm

def create_llm_model(max_tokens: int = 2048):

    # vLLM 모델 인스턴스 생성
    llm = init_chat_model(
        "openai:",
        temperature=0.0,
        top_p=0.1,  # top_p는 (0, 1] 범위여야 하므로 0.1로 설정
        base_url=LLM_BASE_URL,
        api_key=LLM_API_KEY
    )
    return llm

In [51]:
# ============================================================
# 2) 목표 테이블(tb_variable_order_data) 적재용 Row 모델
# ============================================================
class DBRow(BaseModel):
    task_id: int
    fund_code: str
    fund_name: str
    settle_class: Literal["CONFIRMED", "PENDING"]
    order_type: Literal["SUB", "RED"]
    base_date: date
    t_day: Optional[int]
    transfer_amount: Decimal  # DECIMAL(18,2)


# ============================================================
# 3) LLM이 출력하는 "값만(Value-only) 추출" 스키마
# ============================================================
class ExtractedOrder(BaseModel):
    fund_code: str
    fund_name: str
    settle_class: Literal["CONFIRMED", "PENDING"]
    order_type: Literal["SUB", "RED"]
    base_date: Optional[str] = Field(description="YYYY-MM-DD or null")
    t_day: Optional[int] = Field(description="CONFIRMED=0, PENDING=1..N or null")
    transfer_amount: str = Field(description="금액 문자열 (콤마/부호 포함 가능)")

class ExtractionValueOnly(BaseModel):
    orders: List[ExtractedOrder]
    issues: List[str] = Field(default_factory=list)

In [52]:
# ============================================================
# 4) 파싱 유틸 (금액/날짜/파일명 base_date 추정)
# ============================================================
def parse_decimal_amount(s: str) -> Decimal:
    """
    문자열 금액 -> Decimal(18,2)
    - 콤마 제거
    - 괄호 음수 지원
    - 부호 유지
    """
    s = (s or "").strip()
    if not s:
        return Decimal("0.00")

    neg = False
    if s.startswith("(") and s.endswith(")"):
        neg = True
        s = s[1:-1].strip()

    s = re.sub(r"[^\d\-,\.]", "", s)
    if s.startswith("-"):
        neg = True
        s = s[1:]

    s = s.replace(",", "")
    if s == "" or s == ".":
        return Decimal("0.00")

    val = Decimal(s) if "." in s else Decimal(s)
    if neg:
        val = -val
    return val.quantize(Decimal("0.01"))

def abs_decimal(d: Decimal) -> Decimal:
    return (d if d >= 0 else -d).quantize(Decimal("0.01"))

def infer_order_type_from_amount(amount: Decimal) -> Literal["SUB", "RED"]:
    return "SUB" if amount >= 0 else "RED"

def guess_base_date_from_filename(path: str) -> Optional[date]:
    """
    파일명에 yymmdd(예: 250826)가 있으면 2025-08-26으로 추정.
    문서에 base_date가 없을 때만 보조적으로 사용.
    """
    name = Path(path).name
    m = re.search(r"(\d{6})", name)
    if not m:
        return None
    yymmdd = m.group(1)
    yy, mm, dd = int(yymmdd[0:2]), int(yymmdd[2:4]), int(yymmdd[4:6])
    year = 2000 + yy  # 업무 관례상 20xx 가정
    try:
        return date(year, mm, dd)
    except ValueError:
        return None

In [53]:
# ============================================================
# 5) Ingest: PDF/XLS/XLSX -> 정규화 Markdown 생성
# ============================================================
def df_to_markdown(df: pd.DataFrame, max_rows: int = 2000) -> str:
    if df is None:
        return ""
    df = df.fillna("")
    if len(df) > max_rows:
        df = df.head(max_rows)
    return df.to_markdown(index=False)

def extract_excel_to_markdown(xls_path: str, max_rows_per_sheet: int = 2000) -> Tuple[str, List[str]]:
    issues: List[str] = []
    chunks: List[str] = []
    try:
        xls = pd.ExcelFile(xls_path)
    except Exception as e:
        return "", [f"EXCEL_OPEN_FAIL:{type(e).__name__}:{e}"]

    for sheet in xls.sheet_names:
        try:
            # dtype=str 제거: 숫자/날짜 형식 깨짐 완화
            df = pd.read_excel(xls_path, sheet_name=sheet)
            md = df_to_markdown(df, max_rows=max_rows_per_sheet)
            chunks.append(f"\n\n# SHEET {sheet}\n{md}\n")
        except Exception as e:
            issues.append(f"SHEET_READ_FAIL:{sheet}:{type(e).__name__}:{e}")
            continue

    return "\n".join(chunks), issues

def extract_pdf_to_markdown(pdf_path: str, max_pages: int = 50) -> Tuple[str, List[str]]:
    issues: List[str] = []
    chunks: List[str] = []
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for i, page in enumerate(pdf.pages[:max_pages]):
                text = page.extract_text() or ""
                chunks.append(f"\n\n# PAGE {i+1}\n{text}\n")

                # 가능한 경우 표도 추출해 Markdown으로 추가(표 구조 개선)
                try:
                    tables = page.extract_tables() or []
                    for ti, tbl in enumerate(tables):
                        # tbl: List[List[str]] 형태
                        if not tbl or len(tbl) < 2:
                            continue
                        df = pd.DataFrame(tbl[1:], columns=tbl[0])
                        md = df_to_markdown(df, max_rows=2000)
                        chunks.append(f"\n\n## PAGE {i+1} TABLE {ti+1}\n{md}\n")
                except Exception:
                    # 테이블 추출 실패는 무시(텍스트는 이미 확보)
                    pass
    except Exception as e:
        issues.append(f"PDF_OPEN_FAIL:{type(e).__name__}:{e}")
        return "", issues

    return "\n".join(chunks), issues

def ingest_file_to_markdown(path: str) -> Tuple[str, List[str]]:
    ext = Path(path).suffix.lower()
    if ext == ".pdf":
        return extract_pdf_to_markdown(path)
    if ext in [".xlsx", ".xls"]:
        return extract_excel_to_markdown(path)
    return "", [f"UNSUPPORTED_EXT:{ext}"]

In [54]:
# ============================================================
# 6) LangGraph State / Nodes (파일 1개 단위 실행)
# ============================================================
class RState(TypedDict):
    task_id: int
    file_path: str

    markdown_doc: str
    ingest_issues: List[str]

    extracted: Optional[ExtractionValueOnly]
    validation_ok: bool
    validation_report: str
    iterations: int

    db_rows: List[Dict[str, Any]]
    extract_issues: List[str]


def node_ingest(state: RState) -> RState:
    md, issues = ingest_file_to_markdown(state["file_path"])
    if md.strip():
        md = f"\n\n========== FILE: {Path(state['file_path']).name} ==========\n{md}\n"
    
    display(Markdown(md))

    return {**state, "markdown_doc": md, "ingest_issues": issues}

def build_extract_prompt(doc_text: str) -> str:
    # ✅ null 허용은 base_date에만(스키마가 Optional)
    # ✅ 출력 JSON only 강제
    return f"""
당신은 변액일임펀드 설정/해지 지시서에서 tb_variable_order_data 적재용 "주문(이체) 단위" 데이터를 추출합니다.

[출력 규칙 - 최우선]
- 출력은 반드시 JSON만 출력합니다. (설명/마크다운/코드펜스/여분 문자 금지)
- 아래 스키마를 반드시 만족해야 합니다:
  {{
    "orders":[
      {{
        "fund_code":"...",
        "fund_name":"...",
        "settle_class":"CONFIRMED|PENDING",
        "order_type":"SUB|RED",
        "base_date":"YYYY-MM-DD 또는 null",
        "t_day":0 또는 1..N 또는 null,
        "transfer_amount":"숫자 문자열(콤마/부호 포함 가능)"
      }}
    ],
    "issues":[...]
  }}

[추출 목표]
orders[]에는 DB row 후보만 넣습니다.
- 1 order = 1 펀드의 1건 이체(투입/인출)입니다.
- 같은 펀드에 대해 T일/예정일자(T+N)별 금액이 있으면 그만큼 order를 여러 개 만드세요.

[매핑 규칙]
- settle_class:
  - 당일/확정/실행/당일이체/당일투입/당일인출 -> CONFIRMED
  - 예정/청구/예상/T+N -> PENDING
- order_type:
  - 투입/설정/입금/매입 -> SUB
  - 인출/해지/출금/환매 -> RED
  - 금액 부호가 명시된 문서는 음수=RED, 양수=SUB를 우선 적용
- base_date:
  - 기준일/기준일자/T일/결제일/settlement date 등으로 명시된 날짜를 YYYY-MM-DD로 추출
  - 문서에서 찾을 수 없으면 null (이 경우 issues에 BASE_DATE_MISSING을 기록)
- t_day:
  - CONFIRMED는 0
  - PENDING은 (예정일 - base_date) 계산 가능하면 그 값, 아니면 null
- transfer_amount:
  - 문서의 금액을 숫자 문자열로 반환(콤마/부호 포함 가능)

[제외 규칙]
- "합계", "총계", "TOTAL", "summary" 등 요약/합계 행은 절대 orders에 만들지 말 것.

[문서(정규화 markdown)]
{doc_text}
"""

def node_extract_value_only(state: RState) -> RState:
    doc_text = state["markdown_doc"]
    if not doc_text.strip():
        extracted = ExtractionValueOnly(orders=[], issues=["NO_TEXT"])
        return {**state, "extracted": extracted, "extract_issues": extracted.issues}

    llm = create_llm_model(max_tokens=2048)
    extractor = llm.with_structured_output(ExtractionValueOnly)

    prompt = build_extract_prompt(doc_text)
    try:
        extracted: ExtractionValueOnly = extractor.invoke(
            [SystemMessage(_SYSTEM_PROMPT), HumanMessage(prompt)]
        )
    except Exception as e:
        extracted = ExtractionValueOnly(orders=[], issues=[f"LLM_EXTRACT_FAIL:{type(e).__name__}:{e}"])
        return {**state, "extracted": extracted, "extract_issues": extracted.issues}

    # base_date가 null인 orders가 있으면, 파일명 추정(파일 단위 실행이므로 혼선 없음)
    issues = list(extracted.issues)
    guess = guess_base_date_from_filename(state["file_path"])
    for o in extracted.orders:
        if not o.base_date:
            if guess:
                o.base_date = guess.isoformat()
                issues.append("BASE_DATE_FROM_FILENAME")
            else:
                issues.append("BASE_DATE_MISSING")
    extracted.issues = issues

    return {**state, "extracted": extracted, "extract_issues": extracted.issues}

def node_validate(state: RState) -> RState:
    extracted = state.get("extracted")
    if not extracted:
        return {**state, "validation_ok": False, "validation_report": "NO_EXTRACTED"}

    problems: List[str] = []

    for i, o in enumerate(extracted.orders):
        if not o.fund_code or not o.fund_name:
            problems.append(f"ORDER[{i}]:MISSING_FUND")

        if o.settle_class not in ["CONFIRMED", "PENDING"]:
            problems.append(f"ORDER[{i}]:BAD_SETTLE_CLASS:{o.settle_class}")

        if o.order_type not in ["SUB", "RED"]:
            problems.append(f"ORDER[{i}]:BAD_ORDER_TYPE:{o.order_type}")

        # base_date format
        if not o.base_date:
            problems.append(f"ORDER[{i}]:MISSING_BASE_DATE")
        else:
            try:
                datetime.strptime(o.base_date, "%Y-%m-%d")
            except Exception:
                problems.append(f"ORDER[{i}]:BAD_BASE_DATE:{o.base_date}")

        # amount
        try:
            _ = parse_decimal_amount(o.transfer_amount)
        except Exception:
            problems.append(f"ORDER[{i}]:BAD_AMOUNT:{o.transfer_amount}")

        # t_day consistency
        if o.settle_class == "CONFIRMED":
            if o.t_day is not None and o.t_day != 0:
                problems.append(f"ORDER[{i}]:CONFIRMED_TDAY_NOT_ZERO:{o.t_day}")
        if o.settle_class == "PENDING":
            if o.t_day is not None and o.t_day < 1:
                problems.append(f"ORDER[{i}]:PENDING_TDAY_LT1:{o.t_day}")

    if problems:
        return {**state, "validation_ok": False, "validation_report": "\n".join(problems)}

    return {**state, "validation_ok": True, "validation_report": "OK"}

def node_repair(state: RState) -> RState:
    extracted = state.get("extracted")
    if not extracted:
        return state

    llm = create_llm_model(max_tokens=2048)
    extractor = llm.with_structured_output(ExtractionValueOnly)

    prompt = f"""
너는 변액일임펀드 지시서 추출 결과를 "오류 항목만" 수정하는 보정 에이전트다.
아래 validation_report의 문제만 해결하여 orders를 수정하라. 다른 항목은 임의 변경 금지.

[validation_report]
{state["validation_report"]}

[현재 추출 결과(JSON)]
{extracted.model_dump_json()}

[문서(정규화 markdown)]
{state["markdown_doc"]}

[출력 규칙]
- JSON만 출력 (설명/마크다운/코드펜스 금지)
- orders 구조 유지, 문제 항목만 수정
- 합계/총계/요약행을 DB row로 만들지 말 것
"""
    fixed: ExtractionValueOnly = extractor.invoke([SystemMessage(_SYSTEM_PROMPT), HumanMessage(prompt)])

    # 보정 후에도 base_date가 비면 파일명 추정 보강
    issues = list(fixed.issues)
    guess = guess_base_date_from_filename(state["file_path"])
    for o in fixed.orders:
        if not o.base_date:
            if guess:
                o.base_date = guess.isoformat()
                issues.append("BASE_DATE_FROM_FILENAME")
            else:
                issues.append("BASE_DATE_MISSING")
    fixed.issues = issues

    return {**state, "extracted": fixed, "iterations": state["iterations"] + 1, "extract_issues": fixed.issues}

def node_transform_to_db(state: RState) -> RState:
    extracted = state.get("extracted")
    if not extracted:
        return {**state, "db_rows": []}

    rows: List[DBRow] = []
    for o in extracted.orders:
        if not o.base_date:
            # base_date 없으면 적재 불가 → 스킵(issues에 이미 남김)
            continue

        base = datetime.strptime(o.base_date, "%Y-%m-%d").date()
        amt = parse_decimal_amount(o.transfer_amount)

        order_type = o.order_type
        if order_type not in ["SUB", "RED"]:
            order_type = infer_order_type_from_amount(amt)

        t_day = o.t_day
        if o.settle_class == "CONFIRMED":
            t_day = 0
        elif o.settle_class == "PENDING":
            if t_day is not None and t_day < 1:
                t_day = None

        rows.append(
            DBRow(
                task_id=state["task_id"],
                fund_code=o.fund_code.strip(),
                fund_name=o.fund_name.strip(),
                settle_class=o.settle_class,
                order_type=order_type,
                base_date=base,
                t_day=t_day,
                transfer_amount=abs_decimal(amt),  # 방향은 order_type, 금액은 절대값 저장(권장)
            )
        )

    return {**state, "db_rows": [r.model_dump() for r in rows]}

def route_after_validate(state: RState) -> str:
    if state["validation_ok"]:
        return "transform"
    if state["iterations"] >= 2:
        return "transform"
    return "repair"

def build_file_graph():
    g = StateGraph(RState)
    g.add_node("ingest", node_ingest)
    g.add_node("extract", node_extract_value_only)
    g.add_node("validate", node_validate)
    g.add_node("repair", node_repair)
    g.add_node("transform", node_transform_to_db)

    g.add_edge(START, "ingest")
    g.add_edge("ingest", "extract")
    g.add_edge("extract", "validate")
    g.add_conditional_edges("validate", route_after_validate, {
        "repair": "repair",
        "transform": "transform",
    })
    g.add_edge("repair", "validate")
    g.add_edge("transform", END)
    return g.compile()

In [55]:
# ============================================================
# 7) 외부 호출 API: 파일 1개 / 여러개
# ============================================================
def extract_one_file(task_id: int, file_path: str) -> Dict[str, Any]:
    app = build_file_graph()

    init_state: RState = {
        "task_id": task_id,
        "file_path": file_path,
        "markdown_doc": "",
        "ingest_issues": [],
        "extracted": None,
        "validation_ok": False,
        "validation_report": "",
        "iterations": 0,
        "db_rows": [],
        "extract_issues": [],
    }

    out = app.invoke(init_state)

    return {
        "file": Path(file_path).name,
        "db_rows": out.get("db_rows", []),
        "ingest_issues": out.get("ingest_issues", []),
        "extract_issues": out.get("extract_issues", []),
        "validation_report": out.get("validation_report", ""),
    }

def extract_many_files(task_id: int, file_paths: List[str]) -> Dict[str, Any]:
    all_rows: List[Dict[str, Any]] = []
    all_issues: List[str] = []
    per_file: List[Dict[str, Any]] = []

    for p in file_paths:
        res = extract_one_file(task_id, p)
        per_file.append(res)

        # merge rows
        all_rows.extend(res["db_rows"])

        # merge issues with filename prefix
        for x in res["ingest_issues"]:
            all_issues.append(f"{res['file']}:INGEST:{x}")
        for x in res["extract_issues"]:
            all_issues.append(f"{res['file']}:EXTRACT:{x}")
        if res.get("validation_report") and res["validation_report"] != "OK":
            all_issues.append(f"{res['file']}:VALIDATE:{res['validation_report']}")

    return {
        "db_rows": all_rows,
        "issues": all_issues,
        "per_file": per_file,
    }

In [56]:

def visualize_res_result(res: Dict[str, Any]):
    """
    extract_one_file의 결과값 res를 테이블 형태로 시각화하는 함수
    
    Args:
        res: extract_one_file 함수의 반환값 (file, db_rows, ingest_issues, extract_issues, validation_report 포함)
    """
    if not res:
        print("res 변수가 비어있습니다.")
        return
    
    # 파일 정보 표시
    print("=" * 80)
    print(f"📄 파일: {res.get('file', 'N/A')}")
    print("=" * 80)
    print()
    
    # db_rows 테이블 생성
    if res.get('db_rows'):
        db_rows_data = []
        for idx, row in enumerate(res['db_rows'], 1):
            row_data = {'Row #': idx}
            # 각 행의 모든 키-값을 추가
            for key, value in row.items():
                # date와 Decimal 타입을 문자열로 변환
                if isinstance(value, date):
                    row_data[key] = value.isoformat()
                elif isinstance(value, Decimal):
                    row_data[key] = str(value)
                else:
                    row_data[key] = value
            db_rows_data.append(row_data)
        
        if db_rows_data:
            df_db_rows = pd.DataFrame(db_rows_data)
            print("=" * 80)
            print("📊 DB 행 데이터")
            print("=" * 80)
            display(df_db_rows)
            print()
    else:
        print("=" * 80)
        print("📊 DB 행 데이터: 없음")
        print("=" * 80)
        print()
    
    # ingest_issues 표시
    if res.get('ingest_issues'):
        print("=" * 80)
        print("⚠️  Ingest 이슈")
        print("=" * 80)
        for idx, issue in enumerate(res['ingest_issues'], 1):
            print(f"{idx}. {issue}")
        print()
    else:
        print("=" * 80)
        print("✅ Ingest 이슈 없음")
        print("=" * 80)
        print()
    
    # extract_issues 표시
    if res.get('extract_issues'):
        print("=" * 80)
        print("⚠️  Extract 이슈")
        print("=" * 80)
        for idx, issue in enumerate(res['extract_issues'], 1):
            print(f"{idx}. {issue}")
        print()
    else:
        print("=" * 80)
        print("✅ Extract 이슈 없음")
        print("=" * 80)
        print()
    
    # validation_report 표시
    validation_report = res.get('validation_report', '')
    print("=" * 80)
    print("🔍 검증 리포트")
    print("=" * 80)
    if validation_report:
        print(validation_report)
    else:
        print("검증 리포트 없음")
    print()

In [57]:
# ============================================================
# 8) 실행 예시
# ============================================================
# graph 실행

# text 추출 태스트

# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/라이나_250826.xlsx"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/신한라이프_251127.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/신한라이프(2차)_251127.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/신한라이프(액티브)_251127.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/신한라이프(퇴직)_251127.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/카디프_251127.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/하나생명(액티브)_251127.pdf"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/iM라이프_250826.xls"
_document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/ABL_250826.xlsx"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/DB_250826.xlsx"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/KB라이프_250826.xls"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_variable_annuity/KB라이프(액티브)_250826.xls"
# _document_file_path = "/Users/bhkim/20_code_test/documents/sample_overseas_settlement/LS.pdf"

_password = None
# _password = '345678'
res = extract_one_file(1, _document_file_path)




========== FILE: ABL_250826.xlsx ==========


# SHEET 호스트
| 특별계정_VL & VUL 자금 운용 현황                                                     | Unnamed: 1                                         | Unnamed: 2               | Unnamed: 3   | Unnamed: 4            | Unnamed: 5        | Unnamed: 6   | Unnamed: 7   | Unnamed: 8   | Unnamed: 9        | Unnamed: 10       | Unnamed: 11            | Unnamed: 12         | Unnamed: 13     | Unnamed: 14    | Unnamed: 15            | Unnamed: 16         | Unnamed: 17     | Unnamed: 18    | Unnamed: 19            | Unnamed: 20         | Unnamed: 21         | Unnamed: 22           | Unnamed: 23            | Unnamed: 24         | Unnamed: 25         | Unnamed: 26           | Unnamed: 27           | Unnamed: 28         | Unnamed: 29         | Unnamed: 30           | Unnamed: 31           | Unnamed: 32         | Unnamed: 33            |
|:-------------------------------------------------------------------------------------|:---------------------------------------------------|:-------------------------|:-------------|:----------------------|:------------------|:-------------|:-------------|:-------------|:------------------|:------------------|:-----------------------|:--------------------|:----------------|:---------------|:-----------------------|:--------------------|:----------------|:---------------|:-----------------------|:--------------------|:--------------------|:----------------------|:-----------------------|:--------------------|:--------------------|:----------------------|:----------------------|:--------------------|:--------------------|:----------------------|:----------------------|:--------------------|:-----------------------|
| 2025-08-26                                                                           |                                                    |                          |              |                       |                   |              |              |              |                   |                   |                        |                     |                 |                |                        |                     |                 |                |                        |                     |                     |                       |                        |                     |                     |                       |                       |                     |                     |                       |                       |                     |                        |
|                                                                                      |                                                    |                          |              |                       |                   |              |              |              |                   |                   |                        | 10                  | 11              | 14             |                        | 17                  | 18              | 21             |                        | 24                  | 25                  | 28                    |                        | 31                  | 32                  | 35                    |                       | 38                  | 39                  | 42                    |                       |                     |                        |
|                                                                                      |                                                    |                          |              |                       |                   |              |              |              |                   |                   |                        | 12                  | 13              | 15             |                        | 19                  | 20              | 22             |                        | 26                  | 27                  | 29                    |                        | 33                  | 34                  | 36                    |                       | 40                  | 41                  | 43                    |                       |                     |                        |
|                                                                                      | ※ 유동성 일일체크 할것. T일은 당일이며 영업일 기준 |                          |              |                       |                   |              |              |              |                   |                   |                        |                     |                 |                |                        |                     |                 |                |                        |                     |                     |                       |                        |                     |                     |                       |                       |                     |                     |                       |                       |                     |                        |
| 운용지시펀드코드                                                                     | 펀드코드_NG&S                                      | 펀드명                   | 수탁사       | 2025-08-25 00:00:00   | T일 (영업일 기준) |              |              |              |                   |                   |                        | T+1일 (영업일 기준) |                 |                |                        | T+2일 (영업일 기준) |                 |                |                        | T+3일 (영업일 기준) |                     |                       |                        | T+4일 (영업일 기준) |                     |                       |                       | T+5일 (영업일 기준) |                     |                       |                       | 5영업일 순유입 합산 | NAV대비                |
|                                                                                      |                                                    |                          |              |                       |                   |              |              |              |                   |                   |                        |                     |                 |                |                        |                     |                 |                |                        |                     |                     |                       |                        |                     |                     |                       |                       |                     |                     |                       |                       | (예상)              |                        |
|                                                                                      |                                                    |                          |              | NAV                   | 투입금액          | 투입좌수     | 인출금액     | 인출좌수     | 순유입금액 (확정) | 순유입좌수 (확정) | NAV대비                | 투입금액 (예상)     | 인출금액 (예상) | 순유입(예상)   |                        | 투입금액 (예상)     | 인출금액 (예상) | 순유입(예상)   |                        | 투입금액 (예상)     | 인출금액 (예상)     | 순유입(예상)          |                        | 투입금액 (예상)     | 인출금액 (예상)     | 순유입(예상)          |                       | 투입금액 (예상)     | 인출금액 (예상)     | 순유입(예상)          |                       |                     |                        |
| 펀드코드_KASS                                                                        | 펀드코드_NG&S                                      | 펀드명                   | 수탁사       | T-1일(영업일기준) NAV | T일 투입금액      | T일 투입좌수 | T일 인출금액 | T일 인출좌수 | T일 순유입        | T일 순유입        | NAV대비                | T+1일 투입금액      | T+1일 인출금액  | T+1일          | NAV대비                | T+2일 투입금액      | T+2일 인출금액  | T+2일          | NAV대비                | T+3일 펀드변경 투입 | T+3일 펀드변경 인출 | T+3일 펀드변경 순투입 | NAV대비                | T+4일 펀드변경 투입 | T+4일 펀드변경 인출 | T+4일 펀드변경 순투입 | NAV대비               | T+4일 펀드변경 투입 | T+4일 펀드변경 인출 | T+4일 펀드변경 순투입 | NAV대비               |                     |                        |
|                                                                                      |                                                    |                          |              |                       |                   |              |              |              | 확정금액          | 확정좌수          |                        |                     |                 | 순유입예상금액 |                        |                     |                 | 순유입예상금액 |                        |                     |                     |                       |                        |                     |                     |                       |                       |                     |                     |                       |                       |                     |                        |
| C1005                                                                                | C1005                                              | 글로벌리츠(VUL)          | 국민은행     | 3550698912            | 920315            | 478728       | 0            | 0            | 920315            | 478728            | 0.0002591926330023896  | 278894              | 269             | 278625         | 7.847046649276693e-05  | 411910              | 0               | 411910         | 0.00011600814662372589 | 142657              | 0                   | 142657                | 4.017716047899023e-05  | 5047571             | 0                   | 5047571               | 0.0014215711118003124 | 0                   | 0                   | 0                     | 0                     | 6801078             | 0.001915419518398185   |
| G2003                                                                                | G2003                                              | 미국주식인덱스(환오픈형) | 국민은행     | 32661937215           | 13629379          | 6365275      | 0            | 0            | 13629379          | 6365275           | 0.00041728630210398864 | 78409034            | 67024047        | 11384987       | 0.00034857047593525603 | 65093322            | 15684704        | 49408618       | 0.0015127277256937804  | 23224219            | 1688                | 23222531              | 0.0007109967436143086  | 153595981           | 19040300            | 134555681             | 0.004119647898233216  | 1035650             | 0                   | 72853939              | 0.0022305455589003405 | 232201196           | 0.007109229145580549   |
| C100311                                                                              | C1003                                              | 1형 성장형(VUL)          | 국민은행     | 12817066479           | 10935024          | 3237109      | 0            | 0            | 10935024          | 3237109           | 0.0008531612142229569  | 345628              | 23804519        | -23458891      | -0.0018302855055356072 | 22092               | 0               | 22092          | 1.7236393394850862e-06 | 294090              | 0                   | 294090                | 2.2945188002406708e-05 | 0                   | 0                   | 0                     | 0                     | 0                   | 0                   | 0                     | 0                     | -12207685           | -0.0009524554639707585 |
| G2005                                                                                | G2005                                              | 탑픽스                   | 국민은행     | 7187484269            | 0                 | 0            | 2811554      | 2256916      | -2811554          | -2256916          | -0.0003911735865811048 | 6383638             | 0               | 6383638        | 0.000888160274316421   | 232887              | 0               | 232887         | 3.2401740481638886e-05 | 0                   | 0                   | 0                     | 0                      | 6519389             | 160                 | 6519229               | 0.0009070251503878456 | 0                   | 0                   | 0                     | 0                     | 10324200            | 0.0014364135786048007  |
| Total                                                                                | Total                                              | 4                        |              | 56217186875           | 25484718          | 10081112     | 2811554      | 2256916      | 22673164          | 7824196           | 0.000403313741941841   | 85417194            | 90828835        | -5411641       | -9.626310565900226e-05 | 65760211            | 15684704        | 50075507       | 0.0008907508501153545  | 23660966            | 1688                | 23659278              | 0.0004208548900286822  | 165162941           | 19040460            | 146122481             | 0.002599249253167117  | 1035650             | 0                   | 72853939              | 0.0012959371154944509 | 237118789           | 0.0042179056295939925  |
|                                                                                      |                                                    | True                     |              |                       |                   |              |              |              | 22673164          | 7824196           |                        |                     |                 |                |                        |                     |                 |                |                        |                     |                     |                       |                        |                     |                     |                       |                       |                     | 1035650             |                       |                       |                     |                        |
|                                                                                      |                                                    | 호스트                   |              | True                  |                   |              |              |              | True              | True              |                        |                     |                 | True           |                        |                     |                 | True           |                        |                     |                     | True                  |                        |                     |                     | True                  |                       |                     |                     | 0                     |                       |                     |                        |
|                                                                                      |                                                    |                          |              |                       |                   |              |              |              |                   |                   |                        |                     |                 | True           |                        |                     |                 | True           |                        |                     |                     | True                  |                        |                     |                     | True                  |                       |                     |                     | 0                     |                       |                     |                        |
| ▶ 익일 이체예상 금액은 정산손익 및 위험보험료 차감 미반영분임.                       |                                                    |                          |              |                       |                   |              |              |              |                   |                   |                        |                     |                 |                |                        |                     |                 |                |                        |                     |                     |                       |                        |                     |                     |                       |                       |                     |                     |                       |                       |                     |                        |
| ▶ 익일과 익익일의 투입예상금액은 예상금액일 뿐 해당일자에 투입금액이 변동될 수 있음. |                                                    |                          |              |                       |                   |              |              |              |                   |                   |                        |                     |                 |                |                        |                     |                 |                |                        |                     |                     |                       |                        |                     |                     |                       |                       |                     |                     |                       |                       |                     |                        |
|                                                                                      |                                                    |                          |              | 134997301269          |                   |              |              |              |                   |                   |                        |                     |                 |                |                        |                     |                 |                |                        |                     |                     |                       |                        |                     |                     |                       |                       |                     |                     |                       |                       |                     |                        |
|                                                                                      |                                                    |                          |              | 122180234790          |                   |              |              |              |                   |                   |                        |                     |                 |                |                        |                     |                 |                |                        |                     |                     |                       |                        |                     |                     |                       |                       |                     |                     |                       |                       |                     |                        |



In [58]:
visualize_res_result(res)

📄 파일: ABL_250826.xlsx

📊 DB 행 데이터


,Row #,task_id,fund_code,fund_name,settle_class,order_type,base_date,t_day,transfer_amount
0,1,1,C1005,글로벌리츠(VUL),CONFIRMED,SUB,2025-08-26,0,920315.00
1,2,1,C1005,글로벌리츠(VUL),PENDING,SUB,2025-08-26,1,278894.00
2,3,1,C1005,글로벌리츠(VUL),PENDING,RED,2025-08-26,1,269.00
3,4,1,C1005,글로벌리츠(VUL),PENDING,SUB,2025-08-26,2,411910.00
4,5,1,C1005,글로벌리츠(VUL),PENDING,SUB,2025-08-26,3,142657.00
5,6,1,C1005,글로벌리츠(VUL),PENDING,SUB,2025-08-26,4,5047571.00
6,7,1,G2003,미국주식인덱스(환오픈형),CONFIRMED,SUB,2025-08-26,0,13629379.00
7,8,1,G2003,미국주식인덱스(환오픈형),PENDING,SUB,2025-08-26,1,78409034.00
8,9,1,G2003,미국주식인덱스(환오픈형),PENDING,RED,2025-08-26,1,67024047.00
9,10,1,G2003,미국주식인덱스(환오픈형),PENDING,SUB,2025-08-26,2,65093322.00



✅ Ingest 이슈 없음

✅ Extract 이슈 없음

🔍 검증 리포트
OK

